# Bump hunt: inference with the released RACNN-ClassiFlow ensemble

This notebook loads the paper's RACNN-ClassiFlow ensemble (3 of 5 runs, `weights/ensembles.json`) and the bump-hunt test set, and shows for one signal region (SR):

* the input histogram inside the SR window,
* the ensemble posterior $p(\theta_m, \theta_s \mid x)$ and its entropy $\hat H$,
* the classifier probability that signal is present,

for a window that contains a signal and for a background-only window with the same edges. It ends with a quick check against the paper's Table 2 NLL. Everything runs on a CPU in a few minutes.

**Before running:** download the weights and data with `python scripts/download.py` from the repository root.

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

REPO    = os.path.abspath("..")               # repository root (this notebook lives in bumphunt/)
DATA    = os.path.join(REPO, "data")          # Hugging Face dataset repo, see scripts/download.py
WEIGHTS = os.path.join(REPO, "weights")       # Hugging Face model repo
sys.path.insert(0, REPO)
import racnn_inference as ri

_, fd = ri.import_analysis("bumphunt")           # puts bumphunt/ on sys.path; fd = flexibledataset
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Load the test set and the ensemble

Arrays in the HDF5 files are stored normalised, exactly as the models see them; `unnormalize_*` gives physical values. Each row is one SR: a histogram of the events inside the window `[lower_limit, upper_limit]`, the total event count, and (for signal rows) the true signal parameters $\theta = (\theta_m, \theta_s)$.

In [ ]:
test  = fd.load_h5(f"{DATA}/bumphunt/test.h5")
theta = test.unnormalize_output("flow_outputs")                       # true (mean, sigma); NaN where no signal was injected
aux   = {k: test.unnormalize_input(k, test.other_inputs[k]) for k in test.other_inputs}
has_signal = test.mlp_outputs["nevents_sg"] > 0
print(f"{len(test)} SR rows, {has_signal.sum()} with signal")

ensemble = ri.load_ensemble("bumphunt", "racnnclassiflows", "best_model", WEIGHTS, device)
print("ensemble runs:", json.load(open(f"{WEIGHTS}/ensembles.json"))["bumphunt"]["racnnclassiflows"]["best_model"])

## One SR with signal, and the same window without signal

In [ ]:
def show(i, ax_h, ax_p, bins=80):
    x_cnn, x_aux, _, _ = (t.unsqueeze(0).double().to(device) for t in test[i])
    lo, up = aux["lower_limit"][i], aux["upper_limit"][i]
    h = test.unnormalize_input("histogram", test.histograms["histogram"][i])
    ax_h.stairs(h, np.linspace(lo, up, len(h) + 1), fill=True, alpha=0.6)
    ax_h.set(xlabel="x", ylabel="events per bin", title=("signal + background" if has_signal[i] else "background only") + f", window [{lo:.1f}, {up:.1f}]")
    sc = test.output_scalers
    g0, g1, dens, H = ri.posterior_grid(ensemble, test, "bumphunt", x_cnn, x_aux,
                                        (sc["mean"]["min"], sc["mean"]["max"]), (sc["sigma"]["min"], sc["sigma"]["max"]), bins)
    _, p_sig = ri.classifier_probability(ensemble, x_cnn, x_aux)
    ax_p.pcolormesh(g0, g1, dens, shading="auto", cmap="viridis")
    if has_signal[i]:
        ax_p.plot(theta["mean"][i], theta["sigma"][i], "r*", ms=14, label="true $\\theta$"); ax_p.legend(loc="upper right")
    ax_p.set(xlabel=r"$\theta_m$", ylabel=r"$\theta_s$", title=f"$\\hat H$ = {H:.2f},  P(signal) = {float(p_sig):.3f}")

TRUE_M, TRUE_S = 10.0, 3.0                                            # one of the paper's Fig 3 examples
sig = np.where(has_signal & np.isclose(theta["mean"], TRUE_M) & np.isclose(theta["sigma"], TRUE_S))[0]
i_sig = sig[np.argmax(test.mlp_outputs["nevents_sg"][sig])]           # the SR with the most signal events for this theta
same_window = np.isclose(aux["lower_limit"], aux["lower_limit"][i_sig]) & np.isclose(aux["upper_limit"], aux["upper_limit"][i_sig])
i_bkg = np.where(same_window & ~has_signal)[0][0]

fig, ax = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
show(i_sig, ax[0, 0], ax[0, 1]); show(i_bkg, ax[1, 0], ax[1, 1])
plt.show()

## Check against the paper

**Mean test NLL (Table 2).** Mean of $-\log p(\theta_\mathrm{true} \mid x)$ over signal rows, excluding $\theta$ on the edge of the training grid ($\theta_m \in \{2, 28\}$ or $\theta_s \in \{1, 5\}$). The ensemble density is the equal-weight mixture of its members. Expected: RACNN-ClassiFlow $-4.602$, RACNN-Flow $-4.490$, CNN-ClassiFlow $-3.004$, CNN-Flow $-2.986$.

In [ ]:
edge = np.isin(np.round(theta["mean"], 1), [2, 28]) | np.isin(np.round(theta["sigma"], 1), [1, 5])
rows = np.where(~edge)[0]                                             # background rows are kept; only signal rows enter the NLL
loader = DataLoader(Subset(test, rows), batch_size=4096)
for family in ("racnnclassiflows", "racnnflows", "cnnclassiflows", "cnnflows"):
    ens = ri.load_ensemble("bumphunt", family, "best_model", WEIGHTS, device)
    print(f"{family:18s} ensemble NLL: {ri.mean_nll(ens, loader, 'bumphunt', device):.3f}")